In [1]:
import os
os.getcwd()
import pandas as pd
import altair as alt
alt.data_transformers.enable('default', max_rows=None) 


DataTransformerRegistry.enable('default')

In [2]:
# Loading and checking dataset 
df = pd.read_csv('../data/diabetes_01.csv')
# checking
df.head()


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
# Data cleaning 
df['Diabetes_012'] = df['Diabetes_012'].map({
    0.0: 'No Diabetes',
    1.0: 'Prediabetes',
    2.0: 'Diabetes'
})

df['HighBP'] = df['HighBP'].map({
    0.0: 'No High BP',
    1.0: 'High BP'
})

df['Sex'] = df['Sex'].map({
    0.0: 'Female',
    1.0: 'Male'
})

df.head()


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,No Diabetes,High BP,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,Female,9.0,4.0,3.0
1,No Diabetes,No High BP,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,Female,7.0,6.0,1.0
2,No Diabetes,High BP,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,Female,9.0,4.0,8.0
3,No Diabetes,High BP,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,Female,11.0,3.0,6.0
4,No Diabetes,High BP,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,Female,11.0,5.0,4.0


In [ ]:
# Visualization: Relationship between Diabetes Status and High Blood Pressure
# Stacked normalized bar chart showing distribution of HighBP within each Diabetes_012 group

# Define order (from unhealthy to diseased for better visualization effect)

order = ['Diabetes', 'Prediabetes', 'No Diabetes']

chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        y=alt.Y('Diabetes_012:N', sort=order, title='Diabetes Status'),
        x=alt.X('count()', stack='normalize', title='Percentage'),
        color=alt.Color(
            'HighBP:N',
            title='High Blood Pressure',
            scale=alt.Scale(
                domain=['High BP', 'No High BP',],
                range=['#94b4e2', '#e15759']  
            )
        )
    )
    .properties(
        title='People with Diabetes Are More Likely to Have High Blood Pressure',
        width=500,
        height=250
    )
)

chart


In [ ]:
# Pie Chart: Gender Distribution among People with Diabetes

# Preparing Data
base = (
    alt.Chart(df)
    .transform_filter("datum.Diabetes_012 == 'Diabetes'")
    .transform_aggregate(
        count='count()',
        groupby=['Sex']
    )
    .transform_window(
        total='sum(count)',
        frame=[None, None]
    )
    .transform_calculate(
        percentage='datum.count / datum.total * 100'
    )
)
# Working on pie chart
pie = base.mark_arc().encode(
    theta=alt.Theta('count:Q', stack=True),
    color=alt.Color(
        'Sex:N',
        scale=alt.Scale(
            domain=['Male', 'Female'],
            range=['#A7C7E7', '#F4B6C2']
        )
    ),
    order=alt.Order('Sex:N', sort='ascending')
)
# Adding % in pie segments                                                  #ChatGPT: prompt, how to add labels in pie chart
text = base.mark_text(radius=120, size=16, fontWeight='bold').encode(       #ChatGPT suggested mark_text 
    theta=alt.Theta('count:Q', stack=True),                 
    text=alt.Text('percentage:Q', format='.1f'),
    color=alt.value('black'),
    order=alt.Order('Sex:N', sort='ascending')                      
)
# Adding all of them 
chart = (pie + text).properties(
    title='Gender Distribution among People with Diabetes(%)'
)

chart


In [ ]:
# Helper Function to calculate BMIs from data 
def categorize_bmi(bmi):
    """Categorize BMI value into standard weight categories."""
    
    if bmi < 18.5:
        return 'Underweight'
    elif 18.5 <= bmi < 25:
        return 'Normal'
    elif 25 <= bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'
# Preparing Data 

# Create categorical BMI variable
df['BMI_Category'] = df['BMI'].apply(categorize_bmi)

# filter for diabetic patients only 
df_diabetes = df[df['Diabetes_012'] == 'Diabetes']

# Aggregate counts and percentages
agg_df = (
    df_diabetes['BMI_Category']
    .value_counts()
    .rename_axis('BMI_Category')
    .reset_index(name='Count')
)

# Calculate percentage
total = agg_df['Count'].sum()
agg_df['Percentage'] = (agg_df['Count'] / total) * 100

# Prepare charts
chart = (
    alt.Chart(agg_df)
    .mark_bar(size=50)
    .encode(
        x=alt.X(
            'BMI_Category:N',
            sort=['Underweight', 'Normal', 'Overweight', 'Obese'],
            title='BMI Category'
        ),
        y=alt.Y(
            'Percentage:Q',
            title='Percentage of Diabetic Individuals (%)'
        ),
        color=alt.Color(
            'BMI_Category:N',
            title='BMI Category',
            scale=alt.Scale(
                domain=['Underweight', 'Normal', 'Overweight', 'Obese'],
                range=['#A7C7E7', '#8FD694', '#F4C27F', '#E15759']  
            )
        ),
        tooltip=[
            'BMI_Category', 
            alt.Tooltip('Percentage:Q', format='.1f', title='Percentage (%)')
        ]
    )
    .properties(
        title='Percentage of Diabetic Individuals by BMI Category',
        width=500,
        height=300
    )
)

chart


In [ ]:
# I asked ChatGPT to explain how to reshape my dataset 
# so I could compare general, mental, and physical
# health measures by diabetes status in one visualization. It suggested 'melt'
df_long = df.melt(
    id_vars=['Diabetes_012'],                                   
    value_vars=['GenHlth', 'MentHlth', 'PhysHlth'],
    var_name='HealthType',
    value_name='Value'
)

label_map = {
    'GenHlth': 'General Health',
    'MentHlth': 'Mental Health',
    'PhysHlth': 'Physical Health'
}
df_long['HealthType'] = df_long['HealthType'].map(label_map)

# Set x-axis order
order = ['No Diabetes', 'Prediabetes', 'Diabetes']


# Bar chart to compare average health measures by diabetes status.
chart = (
    alt.Chart(df_long)
    .mark_bar()
    .encode(
        x=alt.X(
            'Diabetes_012:O',
            sort=order,
            axis=alt.Axis(title=None)   # hides axis title but keeps tick labels
        ),
        y=alt.Y(
            'mean(Value):Q',
            title='Number of Days Sick (Average)'
        ),
        color=alt.Color(
            'HealthType:N',
            scale=alt.Scale(
                domain=['General Health', 'Mental Health', 'Physical Health'],
                range=['#A7C957','#70C1B3','#FF7E6B']
            ),
            title='Health Measure'
        ),
        # Facet the chart by health measure for easier side-by-side comparison
        column=alt.Column(
            'HealthType:N',
            title=None,
            header=alt.Header(labelAngle=0)  # keeps column labels horizontal   #Asked ChatGPT how to make column labels horizontal, 
        )                                                                       # ChatGPT: labelAngle=0
    )
    .properties(
        title='Average Health Measures by Diabetes Status'
    )
)

chart


In [ ]:
df['Smoker'] = df['Smoker'].map({0: 'Non-Smokers', 1: 'Smokers'})
 
# Base chart with percentage calculation
base = (
    alt.Chart(df)
    .transform_aggregate(
        count='count()',
        groupby=['Smoker']
    )
    .transform_window(
        total='sum(count)',
        frame=[None, None]
    )
    .transform_calculate(
        percentage='datum.count / datum.total * 100'
    )
)

# Pie chart
pie = base.mark_arc().encode(
    theta=alt.Theta('count:Q', stack=True),
    color=alt.Color(
        'Smoker:N',
        title='Smoking Status',
        scale=alt.Scale(
            domain=['Non-Smokers', 'Smokers'],
            range=['#6abf69', '#d73027']
        )
    )
)

# Text labels with percentages
text = base.mark_text(radius=120, size=16, fontWeight='bold').encode(
    theta=alt.Theta('count:Q', stack=True),
    text=alt.Text('percentage:Q', format='.1f'),
    color=alt.value('black'),
    order=alt.Order('Smoker:N', sort='ascending')
)

# Combine pie and text
chart = (pie + text).properties(
    title='Distribution of Smokers vs Non-Smokers Amongst People with Diabetes(%)'
)

chart


In [ ]:
# Map numeric codes to descriptive labels
education_labels = {
    1: 'Only Kindergarten',
    2: 'Elementary',
    3: 'Some high school',
    4: 'High school graduate',
    5: 'Some college/technical',
    6: 'College graduate',
    9: 'Refused'
}

income_labels = {
    1: '<$10k',
    2: '$10k–15k',
    3: '$15k–20k',
    4: '$20k–25k',
    5: '$25k–35k',
    6: '$35k–50k',
    7: '$50k–75k',
    8: '$75k+'
}

# Apply mapping
df['Education_Label'] = df['Education'].map(education_labels)
df['Income_Label'] = df['Income'].map(income_labels)

# Create heatmap 
chart2 = (
    alt.Chart(df)
    .transform_filter("datum.Diabetes_012 == 'Diabetes'")
    .mark_rect()
    .encode(
        x=alt.X(
            'Education_Label:O',
            title='Education Level',
            sort=[
                'Kindergarten', 'Elementary', 'Some high school',
                'High school graduate', 'Some college/technical',
                'College graduate', 'Refused'
            ]
        ),
        y=alt.Y(
            'Income_Label:O',
            title='Income Level',
            sort=[
                '<$10k', '$10k–15k', '$15k–20k', '$20k–25k',
                '$25k–35k', '$35k–50k', '$50k–75k', '$75k+'
            ]
        ),
        color=alt.Color('count()', title='Number of Diabetic Individuals')
    )
    .properties(
        title='Diabetes Distribution by Education and Income Levels',
        width=550,
        height=350
    )
)

chart2
